# Milestone 1, Part B: DDIM sampling and FID on CIFAR-10

Goal: reproduce the DDIM paper's CIFAR-10 result of FID 4.67 with 50 sampling steps.

Outline of the notebook:

1. Setup: check the GPU, get the official DDIM code, install diffusers and clean-fid.
2. Model weights: load a pretrained DDPM CIFAR-10 U-Net, either the EMA weights or google/ddpm-cifar10-32.
3. Sampling settings: batch size, seeds, and the quadratic timestep sequence from the DDIM paper.
4. Output folders on Google Drive and on the local disk.
5. Generate 50,000 images in batches of 250, saved to Drive as we go.
6. Compute FID against the CIFAR-10 training set, in clean and legacy_tensorflow modes.
7. Repeat the FID evaluation for the google/ddpm-cifar10-32 weights.

## 1. Setup

Check which GPU Colab assigned. This notebook was run on a Tesla T4.

In [ ]:
!nvidia-smi

Mon Oct 26 11:51:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Clone the official DDIM repository (ermongroup/ddim). We only use its generalized_steps function, which implements the DDIM update rule exactly as in the paper's code.

In [ ]:
!git clone https://github.com/ermongroup/ddim.git /content/ddim

Cloning into '/content/ddim'...
remote: Enumerating objects: 66, done.
remote: Counting objects: 100% (53/53), done.
remote: Compressing objects: 100% (30/30), done.
remote: Total 66 (delta 25), reused 23 (delta 23), pack-reused 13 (from 1)
Receiving objects: 100% (66/66), 30.58 KiB | 5.10 MiB/s, done.
Resolving deltas: 100% (25/25), done.


Add the cloned repository to the Python path so that functions.denoising can be imported later.

In [ ]:
import sys; sys.path.insert(0, "/content/ddim")

Confirm the PyTorch version and that CUDA is available.

In [ ]:
import torch; print(torch.__version__, torch.cuda.is_available())

2.11.0+cu130 True


Install the libraries:

- diffusers provides the pretrained U-Net and its noise schedule.
- clean-fid computes FID, both in its modern clean mode and in a legacy mode that matches the original TensorFlow implementation.

torchao is uninstalled to avoid a version conflict with diffusers on this Colab image.

In [ ]:
!pip install --upgrade diffusers[torch]
!pip install clean-fid
!pip uninstall --yes torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 26.3 MB/s  0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.41.0
    Uninstalling diffusers-0.41.0:
      Successfully uninstalled diffusers-0.41.0
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


Record the installed versions for reproducibility.

In [ ]:
!pip show clean-fid
!pip show diffusers

Name: clean-fid
Version: 0.1.35
Summary: FID calculation in PyTorch with proper image resizing and quantization steps
Home-page: https://github.com/GaParmar/clean-fid
Author: Gaurav Parmar
Author-email: gparmar@andrew.cmu.edu
License: UNKNOWN
Location: /usr/local/lib/python3.13/dist-packages
Requires: numpy, pillow, requests, scipy, torch, torchvision, tqdm
Required-by: 
Name: diffusers
Version: 0.42.0
Summary: State-of-the-art diffusion in PyTorch.
Home-page: https://github.com/huggingface/diffusers
Author: The Hugging Face team (past and future) with the help of all our contributors (https://github.com/huggingface/diffusers/graphs/contributors)
Author-email: diffusers@huggingface.co
License: Apache 2.0 License
Location: /usr/local/lib/python3.13/dist-packages
Requires: filelock, huggingface-hub, importlib_metadata, numpy, Pillow, regex, requests, safetensors
Required-by: 


Mount Google Drive. The generated batches are saved there so that progress survives a Colab disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## 2. Model weights

The required checkpoint, google/ddpm-cifar10-32, contains the raw training weights, not their exponential moving average (EMA). The DDPM and DDIM papers report FID with the EMA weights. Users of the raw checkpoint report FID around 12 instead of the published numbers (see https://huggingface.co/google/ddpm-cifar10-32/discussions/10).

In that discussion, the Diff-Pruning authors converted the official EMA checkpoint (from pesser/pytorch_diffusion) to diffusers format. This cell downloads it and unzips it to /content/ddpm_ema_cifar10.

In [ ]:
!wget -q -O /content/ddpm_ema_cifar10.zip https://github.com/VainF/Diff-Pruning/releases/download/v0.0.1/ddpm_ema_cifar10.zip
!unzip -q -o /content/ddpm_ema_cifar10.zip -d /content

Imports. generalized_steps comes from the cloned DDIM repository, everything else from standard packages.

In [ ]:
from diffusers import DDPMPipeline, DDIMPipeline
from pathlib import Path
import os
import glob
import shutil
import matplotlib.pyplot as plt
import time
import numpy as np
from PIL import Image
from tqdm import tqdm
from cleanfid import fid
from functions.denoising import generalized_steps


Check that the EMA checkpoint was unpacked where we expect it.

In [ ]:
emr_path = Path("/content/ddpm_ema_cifar10")
assert emr_path.exists()

Choose which weights to use with weights_origin:

- 'ema': the converted official EMA checkpoint (used for the main result)
- 'google': google/ddpm-cifar10-32 from the Hugging Face Hub (the checkpoint named in the assignment)

The DDIMPipeline is only used as a container. We take its U-Net and its beta schedule and do the sampling ourselves in section 5.

The warning about a missing .safetensors file is expected for the EMA checkpoint, which is stored in PyTorch's pickle format instead.

In [ ]:
weights_origin = 'ema' # 'google'
model_id = {"google": "google/ddpm-cifar10-32", "ema": "/content/ddpm_ema_cifar10"}[weights_origin]

device = 'cuda'
pipeline = DDIMPipeline.from_pretrained(model_id).to(device)
pipeline.set_progress_bar_config(disable=True)


Loading pipeline components...:   0%|          | 0/2 [00:00<?, ?it/s]

An error occurred while trying to fetch /content/ddpm_ema_cifar10/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /content/ddpm_ema_cifar10/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Sanity check of the loaded model and scheduler. These should match the DDPM paper: 1000 training timesteps and a linear beta schedule from 0.0001 to 0.02.

The timestep spacing shows 'leading', the diffusers default. That setting is not used here, because we pass our own timestep sequence to generalized_steps (see section 3).

In [ ]:
print(pipeline.unet.num_parameters()/ 1e6, 'million parameters')
print('sample size: ',pipeline.unet.config.sample_size, '\ninput channels: ',pipeline.unet.config.in_channels)
print("training timesteps: ", pipeline.scheduler.config['num_train_timesteps'])
print("beta start: ", pipeline.scheduler.config['beta_start'])
print("beta end: ", pipeline.scheduler.config['beta_end'])
print("beta schedule: ", pipeline.scheduler.config['beta_schedule'])
print("timestep spacing: ", pipeline.scheduler.config['timestep_spacing'])
print("clip sample: ", pipeline.scheduler.config['clip_sample'])
print("set_alpha_to_one: ", pipeline.scheduler.config['set_alpha_to_one'])
print("alphas_cumprod: ", pipeline.scheduler.alphas_cumprod[0:3], "...", pipeline.scheduler.alphas_cumprod[-3:])

35.746307 million parameters
sample size:  32 
input channels:  3
training timesteps:  1000
beta start:  0.0001
beta end:  0.02
beta schedule:  linear
timestep spacing:  leading
clip sample:  True
set_alpha_to_one:  True
alphas_cumprod:  tensor([0.9999, 0.9998, 0.9996]) ... tensor([4.2022e-05, 4.1182e-05, 4.0358e-05])


## 3. Sampling settings

- N_IMAGES: 50,000 samples, the standard number for FID.
- BATCH: 250 images per batch, so 200 batches in total.
- SEED: base seed. Batch b uses seed SEED + b, so every batch is reproducible on its own.
- STEPS: 50 DDIM steps, as in the target result.

In [ ]:
N_IMAGES = 50_000
BATCH    = 250
SEED     = 1000
STEPS    = 50
N_TOTAL = 50_000
n_batches = N_TOTAL // BATCH


Two small helpers:

- model(x, t) adapts the diffusers U-Net to the interface generalized_steps expects. The U-Net returns an output object, while generalized_steps wants the plain noise prediction tensor, which is its .sample attribute.
- to_uint8(x) converts the sampler output from floats in [-1, 1] with shape (B, 3, 32, 32) to 8-bit images with shape (B, 32, 32, 3), ready to save as PNG.

In [ ]:
def model(x, t):
    return pipeline.unet(x, t).sample

def to_uint8(x):
    x = ((x + 1) / 2).clamp(0, 1)
    return (x * 255).round().to(torch.uint8).permute(0, 2, 3, 1).cpu().numpy()

Quadratic timestep sequence.

The diffusers DDIMScheduler only offers linear spacings (leading, linspace, trailing). The DDIM paper's CIFAR-10 experiments use a quadratic spacing, which places more steps at low noise levels, where fine detail is formed.

This line uses the same formula as the 'quad' skip type in the official DDIM code: 50 points evenly spaced between 0 and sqrt(800), then squared and truncated to integers.

The first two entries are both 0 because of the integer truncation. The original code does the same, so it is kept as is.

betas is the linear DDPM noise schedule, taken from the pipeline's scheduler.

In [ ]:
betas = pipeline.scheduler.betas.to("cuda")

seq = [int(s) for s in np.linspace(0, np.sqrt(1000 * 0.8), STEPS) ** 2]

print(len(seq), seq[:5], "...", seq[-3:])

50 [0, 0, 1, 2, 5] ... [736, 767, 800]


## 4. Output folders

- save_batches_path (on Drive): one .npy file per batch of 250 images, under Milestone_1_b/{weights_origin}/run_0/ddim50_batches. This is the permanent copy.
- png_img_50k_path (local Colab disk): the 50,000 images as individual PNG files, which is the input format clean-fid expects. Writing 50,000 small files to Drive is very slow, so these go on the local disk and are rebuilt from the .npy batches when needed. The folder is emptied first so that no stale images get mixed in.

In [ ]:
output_path = Path('/content/drive/MyDrive/deep_learning_project/Milestone_1_b')
run_id = 'run_0'
save_batches_path = output_path/f'{weights_origin}'/run_id/'ddim50_batches'
Path.mkdir(save_batches_path, exist_ok=True, parents=True)

png_img_50k_path = Path(f'/content/{weights_origin}/imgs_50k_pngs')
shutil.rmtree(png_img_50k_path, ignore_errors=True)
Path.mkdir(png_img_50k_path, exist_ok=True, parents=True)


## 5. Generate 50,000 images

For each of the 200 batches:

1. Skip the batch if its .npy file already exists on Drive. This makes the loop resumable after a disconnect.
2. Draw the starting noise from a generator seeded with SEED + b.
3. Run deterministic DDIM sampling (eta = 0) along the quadratic sequence. generalized_steps returns all intermediate states, and the last one is the final image batch.
4. Save the batch to a temporary file, then rename it to its final name. A batch interrupted halfway through saving therefore never looks complete.

The output below is from a resumed session in which all 200 batches had already been generated.

In [ ]:
for b in tqdm(range(n_batches)):

  batch_b_path = save_batches_path/f'batch_{b:04d}.npy'

  if batch_b_path.exists():
    print(f'batch {b+1}/{n_batches} already exists')
    continue

  t0 = time.time()
  g = torch.Generator(device='cuda').manual_seed(SEED + b)

  x = torch.randn((BATCH, 3, 32, 32), generator=g, device="cuda")
  xs, _ = generalized_steps(x, seq, model, betas, eta=0.0)
  imgs = to_uint8(xs[-1])
  del xs



  tmp = batch_b_path.with_name(f'batch_{b:04d}_tmp.npy')
  np.save(tmp, imgs)
  tmp.replace(batch_b_path)
  print(f'batch {b+1}/{n_batches}   {time.time()-t0:.1f}s')

100%|██████████| 200/200 [00:00<00:00, 226.17it/s]

batch 1/200 already exists
batch 2/200 already exists
batch 3/200 already exists
batch 4/200 already exists
batch 5/200 already exists
batch 6/200 already exists
batch 7/200 already exists
batch 8/200 already exists
batch 9/200 already exists
batch 10/200 already exists
batch 11/200 already exists
batch 12/200 already exists
batch 13/200 already exists
batch 14/200 already exists
batch 15/200 already exists
batch 16/200 already exists
batch 17/200 already exists
batch 18/200 already exists
batch 19/200 already exists
batch 20/200 already exists
batch 21/200 already exists
batch 22/200 already exists
batch 23/200 already exists
batch 24/200 already exists
batch 25/200 already exists
batch 26/200 already exists
batch 27/200 already exists
batch 28/200 already exists
batch 29/200 already exists
batch 30/200 already exists
batch 31/200 already exists
batch 32/200 already exists
batch 33/200 already exists
batch 34/200 already exists
batch 35/200 already exists
batch 36/200 already exists
b

## 6. FID for the EMA weights

First, unpack the 200 .npy batches into 50,000 PNG files and check that exactly 50,000 images were found.

Then compute FID against the CIFAR-10 training set statistics that clean-fid provides, in two modes:

- clean: resizes images to 299x299 with a properly antialiased filter. This is the more correct measurement, but it gives different numbers from older papers.
- legacy_tensorflow: reproduces the original TensorFlow Inception pipeline used in the DDPM and DDIM papers. This is the number to compare with the paper's 4.67.

Result: clean 5.59, legacy_tensorflow 4.72, within 0.05 of the paper.

In [ ]:
files = sorted(f for f in glob.glob(str(save_batches_path / "batch_*.npy")) if not f.endswith("_tmp.npy"))
k = 0
for f in tqdm(files):
    for img in np.load(f):
        Image.fromarray(img).save(png_img_50k_path/f"{k:05d}.png")
        k += 1
assert k == N_IMAGES, f"found {k} images, expected {N_IMAGES}"

for mode in ["clean", "legacy_tensorflow"]:
    score = fid.compute_fid(str(png_img_50k_path), dataset_name="cifar10", dataset_res=32,
                            dataset_split="train", mode=mode)
    print(f"FID ({mode}): {score:.2f}")

100%|██████████| 200/200 [00:28<00:00,  7.11it/s]


compute FID of a folder with cifar10 statistics
downloading statistics to /usr/local/lib/python3.13/dist-packages/cleanfid/stats/cifar10_clean_train_32.npz


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Found 50000 images in the folder /content/ema/imgs_50k_pngs


FID imgs_50k_pngs :   0%|          | 0/1563 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 12 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
FID imgs_50k_pngs : 100%|██████████| 1563/1563 [04:28<00:00,  5.83it/s]


FID (clean): 5.59
compute FID of a folder with cifar10 statistics
downloading statistics to /usr/local/lib/python3.13/dist-packages/cleanfid/stats/cifar10_legacy_tensorflow_train_32.npz
Found 50000 images in the folder /content/ema/imgs_50k_pngs


FID imgs_50k_pngs : 100%|██████████| 1563/1563 [03:51<00:00,  6.75it/s]


FID (legacy_tensorflow): 4.72


## 7. FID for the google/ddpm-cifar10-32 weights

The same evaluation for the non-EMA checkpoint. Its batches were generated in an earlier session by setting weights_origin = 'google' in the loading cell of section 2 and running sections 3 to 5.

This cell only points the paths to that run's folders.

In [ ]:
weights_origin = 'google'
run_id = 'run_0'
save_batches_path = output_path/f'{weights_origin}'/run_id/'ddim50_batches'
Path.mkdir(save_batches_path, exist_ok=True, parents=True)
png_img_50k_path = Path(f'/content/{weights_origin}/imgs_50k_pngs')
shutil.rmtree(png_img_50k_path, ignore_errors=True)
Path.mkdir(png_img_50k_path, exist_ok=True, parents=True)

Unpack the google batches and compute both FID modes, exactly as in section 6.

In [21]:
files = sorted(f for f in glob.glob(str(save_batches_path / "batch_*.npy")) if not f.endswith(".tmp.npy"))
k = 0
for f in tqdm(files):
    for img in np.load(f):
        Image.fromarray(img).save(png_img_50k_path/f"{k:05d}.png")
        k += 1
assert k == N_IMAGES, f"found {k} images, expected {N_IMAGES}"

for mode in ["clean", "legacy_tensorflow"]:
    score = fid.compute_fid(str(png_img_50k_path), dataset_name="cifar10", dataset_res=32,
                            dataset_split="train", mode=mode)
    print(f"FID ({mode}): {score:.2f}")

100%|██████████| 200/200 [00:26<00:00,  7.57it/s]


compute FID of a folder with cifar10 statistics
Found 50000 images in the folder /content/google/imgs_50k_pngs


FID imgs_50k_pngs : 100%|██████████| 1563/1563 [04:30<00:00,  5.79it/s]


FID (clean): 11.89
compute FID of a folder with cifar10 statistics
Found 50000 images in the folder /content/google/imgs_50k_pngs


FID imgs_50k_pngs : 100%|██████████| 1563/1563 [03:51<00:00,  6.75it/s]


FID (legacy_tensorflow): 9.79


## Summary

50,000 samples, 50 DDIM steps, quadratic spacing, FID against CIFAR-10 train:

| Weights | FID (clean) | FID (legacy_tensorflow) |
|---|---|---|
| EMA (converted official) | 5.59 | 4.72 |
| google/ddpm-cifar10-32 (non-EMA) | 11.89 | 9.79 |
| DDIM paper | | 4.67 |

- With the EMA weights, the legacy FID of 4.72 reproduces the paper's 4.67. The small remaining gap is within what you would expect from a different set of random seeds and from rounding the samples to 8-bit images.
- The non-EMA google checkpoint scores about twice as high. This matches the reports in the Hugging Face discussion and confirms that EMA weights are needed to reproduce the published result.
- Clean FID is higher than legacy FID for both checkpoints because of the different resizing. Only legacy FID is comparable to numbers published with the original implementation.